# 策略概述

**SSD Basic** 是本研究一切策略的**基礎原型**，
以 Zhu (2024) 記載的 Gatev et al. (2006) 基準模型規格為原型復刻：

1. 價格正規化為**累積總回報指數**（首日 = 1.0，即「首日投入 $1 的價值路徑」）
2. **全市場**兩兩計算均方差距離 $D_{i,j}$（不限制同產業）
3. 依距離升序**直接取前 `top_n` 對**——不做共整合、半衰期、Hurst 等任何額外統計過濾
4. 對沖比例固定 $\beta = 1$（等金額對沖，無回歸估計）
5. 記錄形成期價差標準差 $s_{i,j}$，作為交易期 2 倍標準差開倉門檻的基準

（2026-07-06 重寫：移除先前版本自加的三道統計過濾與同產業限制，回歸論文原始規格）。


# 參考文獻與引用對應


## 文獻 1：Gatev, Goetzmann & Rouwenhorst (2006)

> Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative value arbitrage rule. *Review of Financial Studies, 19*(3), 797–827.

**參考部分**：

- 基準模型的**完整規格**：12 個月形成期以標準化價格（累積回報指數）的距離平方和選取
  前 $M = 20$ 對；6 個月交易期在價差超過形成期 2 倍標準差時開倉、收斂時平倉
- 純距離選取——原始模型**不含任何共整合或統計檢定步驟**

**為何參考**：

- 本策略即此基準模型的復刻：正規化方式、距離定義、選取規則、$\beta = 1$ 等金額對沖
  全部按原文重現，作為研究中一切延伸策略的原型與比較基底



## 文獻 2：Zhu (2024)

> Zhu, X. (2024). Examining Pairs Trading Profitability. Senior Essay, Department of Economics, Yale University.

**參考部分**（第 3.1 節 Methodology，本實作直接對齊的規格來源）：

- 正規化：$P_{s,t} = P'_{s,t} / P'_{s,0}$（首日 = 1，$1 投資價值路徑；total price index 含股息）
- 距離：$D_{i,j} = \frac{1}{T}\sum_t (P_{i,t} - P_{j,t})^2$（全市場 $N(N-1)/2$ 對）
- 形成期價差變異 $s^2_{i,j}$（式 (1)，population 口徑）
- 交易規則：$|P_{i,t} - P_{j,t}| > 2 s_{i,j}$ 開倉、價差符號翻轉平倉、期末強平、可多次往返交易
- 以 2003–2023 資料驗證此規格年化超額報酬 6.2%、Sharpe 1.35

**為何參考**：

- 原論文（1962–2002 資料）的規格經此複製研究以近二十年資料重新驗證仍有效；
  本實作的公式與流程**逐條對齊其第 3.1 節**（距離含 $1/T$ 縮放、$s_{i,j}$ 用 population 標準差）



# 各階段行為

策略在每個滾動形成窗（252 交易日，每 21 日滾動）內依序執行以下四個階段。


## 階段 1：累積總回報指數正規化

對形成窗內每支股票 $s$，以首日價格歸一化：

$$P_{s,t} = \frac{P'_{s,t}}{P'_{s,0}}, \qquad P_{s,0} = 1.0$$

（首日價格 $\le 10^{-8}$ 時分母以 1.0 替代，防除零。）

**意義**：$P_{s,t}$ 即「形成期首日投入 $1 的累積價值」——
兩股指數序列的差即**等金額投資下的相對報酬差**，
距離小 = 等金額投資的價值路徑幾乎重合。

**資料口徑說明**：本平台使用 Tiingo 調整後收盤價（已含股息與拆股調整），
對應原論文的 total return index 口徑。


## 階段 2：全市場均方差距離計算

對**全市場**（當期全部有效 S&P 500 歷史成分股，不分產業）的 $N(N-1)/2$ 對股票：

$$D_{i,j} = \frac{1}{T}\sum_{t=1}^{T} \left(P_{i,t} - P_{j,t}\right)^2$$

- 以 `scipy.spatial.distance.pdist`（`sqeuclidean`）一次算完距離矩陣，再除以 $T$ 對齊論文公式
  （$1/T$ 為常數縮放，不影響排序）
- **不限制同產業**：原論文在全市場搜尋最近鄰配對，產業資訊僅事後記錄
  （`Sector_A/B` 欄位；兩腳異產業時 `Sector` 記為 `CrossSector`）


## 階段 3：純距離選取（無統計過濾）

全部配對依 $D_{i,j}$ **升序直接取前 `top_n` 對**。

原論文的基準模型不含共整合檢定、半衰期或 Hurst 過濾——
配對品質完全由「形成期價值路徑的接近程度」決定。
本實作忠實復刻此設計，**不添加任何額外過濾機制**。

對每組入選配對計算形成期價差統計量（Zhu 2024 式 (1)，population 口徑）：

$$\epsilon_t = P_{A,t} - P_{B,t}, \qquad
\mu_\epsilon = \frac{1}{T}\sum_t \epsilon_t, \qquad
s_{A,B} = \sqrt{\frac{1}{T}\sum_t (\epsilon_t - \mu_\epsilon)^2}$$


## 階段 4：參數輸出與交易期銜接

| 欄位 | 內容 | 交易期用途 |
| :--- | :--- | :--- |
| `Ticker_A` / `Ticker_B` | 配對股票代碼 | 建倉標的 |
| `Sector`（`Sector_A/B`） | 產業記錄（可跨產業） | 結果分析 |
| `SSD` / `Rank` | 距離值／名次 | 記錄用 |
| `Hedge_Ratio` | 恆為 1.0 | 等金額對沖 |
| `Spread_Mean` / `Spread_Std` | $\mu_\epsilon$／$s_{A,B}$ | 開倉門檻基準 |
| `First_Price_A` / `First_Price_B` | 形成期首日價格 | 交易期重建累積回報指數 |

**交易期 spread 重建**（沿用形成期同一座標，路徑 B1）：

$$P_{i,t} = \frac{P'_{i,t}}{\texttt{First\_Price}_i}, \qquad
\text{Spread}_t = P_{A,t} - P_{B,t}, \qquad
Z_t = \frac{\text{Spread}_t - \mu_\epsilon}{s_{A,B}}$$

$|Z_t| > 2$ 開倉即對應原論文「價差超過形成期 2 倍標準差」的開倉條件；
$Z$ 穿越 0 平倉對應「價差收斂」；平倉後可再次進場（多次往返交易）。


## 與原論文的已知實作差異

以下差異來自共用交易引擎與資料範圍，記錄供解讀結果時參考：

| 項目 | 原論文（Gatev／Zhu） | 本實作 |
| :--- | :--- | :--- |
| 開倉時點 | 偏離**次日**開倉（wait-one-day，抑制買賣價差反彈） | 偏離**當日**收盤開倉 |
| 平倉條件 | 價差**符號翻轉**（$P_A - P_B$ 穿越 0） | $Z$ 穿越 0（= 價差穿越形成期均值 $\mu_\epsilon$；$\mu_\epsilon \approx 0$ 時兩者近似） |
| 股票池 | CRSP 全市場（Zhu：13,386 檔） | S&P 500 歷史成分股 |
| 交易成本 | 報酬計算後另行敏感性分析 | 每筆進出場即扣 0.2% 單程摩擦 |
| 除牌處理 | 除牌日平倉 | 價格序列終止時依期末強平處理 |

形成期選取邏輯（本筆記本的範圍）為**完全復刻**；上表差異均屬交易期執行層。


# 參數總表

| 參數 | 值 | 說明 |
| :--- | :---: | :--- |
| 形成窗長度 $T$ | 252 交易日 | 對應原論文 12 個月 |
| 交易期長度 | 126 交易日 | 對應原論文 6 個月 |
| 滾動步長 | 21 交易日 | 對應原論文逐月重啟 |
| `top_n` | 網格 [1, 3, 5, 10, 20] | 原論文基準 $M = 20$ |
| 搜尋範圍 | 全市場 | 不限制同產業 |
| 對沖比例 $\beta$ | 固定 1.0 | 等金額對沖 |
| 統計過濾 | **無** | 純距離排序（復刻原論文） |
| 開倉門檻 | $|Z| > 2$ | 形成期 2 倍標準差 |
